# Where the field actually stands

**Paper:** [https://arxiv.org/abs/2309.03409](https://arxiv.org/abs/2309.03409)  
**Authors:** Chengrun Yang, Xuezhi Wang, Yifeng Lu, Hanxiao Liu, Quoc V. Le, Denny Zhou, Xinyun Chen  
**Repository:** [https://github.com/google-deepmind/opro](https://github.com/google-deepmind/opro)  
**License:** Apache-2.0  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-08 03:12 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/google-deepmind/opro
%cd opro
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
import os, json, re
os.makedirs("/kaggle/working", exist_ok=True)

!pip install -q transformers accelerate datasets

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# OPRO is a prompt-optimization method; the paper's headline numbers were obtained
# with PaLM 2-L (proprietary, deprecated) as scorer. To produce a real signal on a
# single T4, we measure the *effect* of the published OPRO-optimized instruction
# vs the human "Let's think step by step." baseline using a small open instruct LM.
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()

HUMAN = "Let's think step by step."
OPRO  = "Take a deep breath and work on this problem step-by-step."

@torch.no_grad()
def gen(prompt, max_new=400):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

def last_num(s):
    s = s.replace(",", "").replace("$", "")
    nums = re.findall(r"-?\d+(?:\.\d+)?", s)
    if not nums: return None
    try: return float(nums[-1])
    except: return None

def gold_gsm8k(ans):
    m = re.search(r"####\s*(-?\d+(?:\.\d+)?)", ans.replace(",", ""))
    return float(m.group(1)) if m else None

# ---------- GSM8K ----------
gsm = load_dataset("gsm8k", "main", split="test").select(range(100))

def eval_gsm(suffix):
    correct = 0
    for ex in gsm:
        p = f"Question: {ex['question']}\n\n{suffix}\n\nEnd with 'Final Answer: <number>'."
        out = gen(p, max_new=400)
        tail = out.split("Final Answer:")[-1] if "Final Answer:" in out else out.split("answer is")[-1]
        pred = last_num(tail) if tail else last_num(out)
        gold = gold_gsm8k(ex["answer"])
        if pred is not None and gold is not None and abs(pred - gold) < 1e-3:
            correct += 1
    return correct / len(gsm)

print("GSM8K human...")
acc_h_g = eval_gsm(HUMAN)
print("  acc =", acc_h_g)
print("GSM8K opro...")
acc_o_g = eval_gsm(OPRO)
print("  acc =", acc_o_g)

# ---------- BBH (movie_recommendation, the task highlighted in the paper) ----------
try:
    bbh = load_dataset("lukaemon/bbh", "movie_recommendation", split="test").select(range(50))
except Exception:
    bbh = load_dataset("maveriq/bigbenchhard", "movie_recommendation", split="train").select(range(50))

def eval_bbh(suffix):
    correct = 0
    for ex in bbh:
        p = f"{ex['input']}\n\n{suffix}\n\nReply with only the letter of the correct option after 'Answer:'."
        out = gen(p, max_new=300)
        tail = out.split("Answer:")[-1] if "Answer:" in out else out
        m = re.search(r"\(([A-Ea-e])\)", tail) or re.search(r"\b([A-Ea-e])\b", tail)
        pred = m.group(1).upper() if m else None
        gold = ex["target"].strip().strip("()").upper()
        if pred and pred == gold:
            correct += 1
    return correct / len(bbh)

print("BBH human...")
acc_h_b = eval_bbh(HUMAN)
print("  acc =", acc_h_b)
print("BBH opro...")
acc_o_b = eval_bbh(OPRO)
print("  acc =", acc_o_b)

metrics = {
    "accuracy_gsm8k": round(acc_o_g * 100, 2),
    "improvement_over_human_prompts_gsm8k": round((acc_o_g - acc_h_g) * 100, 2),
    "improvement_over_human_prompts_bbh":   round((acc_o_b - acc_h_b) * 100, 2),
}
with open("/kaggle/working/metrics.json", "w") as f:
    json.dump(metrics, f)
print(metrics)

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
